# OthelloRL Sweep Agent
Connects to the existing wandb sweep and runs trials on Colab hardware.

In [ ]:
# 1. Clone the repo
!git clone https://github.com/sanasuri101/OthelloRL.git
%cd OthelloRL

In [ ]:
# 2. Install ONLY what Colab doesn't have — do NOT touch numpy, torch, or gymnasium
# Colab pre-installs numpy 2.x, torch, gymnasium — reinstalling them corrupts binaries

# Fix wandb binary (force-reinstall gets wandb-core)
!pip install -q --force-reinstall wandb

# Install pufferlib against whatever numpy is already installed
!pip install -q pufferlib --no-build-isolation

# Build the C extension without raylib
%cd othello
!make NO_RENDER=1
%cd ..

In [ ]:
# 3. Verify everything imports correctly
import sys, os
sys.path.insert(0, '/content/OthelloRL')
sys.path = [p for p in sys.path if not p.endswith('/othello')]

import torch
import pufferlib
import wandb
print(f'torch: {torch.__version__}')
print(f'pufferlib: {pufferlib.__version__}')
print(f'wandb: {wandb.__version__}')
from othello import binding
print(f'CUDA available: {torch.cuda.is_available()}')
print('binding: OK')

In [ ]:
# 4. Log in to wandb using Colab secret
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["WANDB_AGENT_DISABLE_FLAPPING"] = "true"
os.chdir('/content/OthelloRL')

wandb.login()

In [ ]:
# 5. Create a new sweep (run only once — skip if you already have a sweep ID)
!wandb sweep /content/OthelloRL/sweep.yaml

In [ ]:
# 6. Run 6 parallel agents in threads — each runs 5 trials = 30 trials total
# Uses H100 GPU via --train.device cuda for ~10x speedup vs CPU
import threading, sys

SWEEP_ID = "sriramanasuri-georgia-institute-of-technology/Connect4RL-othello/ux3575ja"

def run_agent():
    sys.argv = [
        'train.py',
        '--wandb',
        '--wandb_project', 'Connect4RL-othello',
        '--train.total_timesteps', '10000000',
        '--train.eval_interval', '500000',
        '--train.device', 'cuda',
        '--curriculum.phase_4_fraction', '0.60',
        '--curriculum.phase_5_fraction', '0.60',
    ]
    from othello.train import train
    def run_trial():
        train()
    wandb.agent(sweep_id=SWEEP_ID, function=run_trial, count=5)

threads = [threading.Thread(target=run_agent) for _ in range(6)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("All 30 trials complete.")